# CCUS private source acquisition
Generated from canonical Git Python modules. Run all only in your authorized Colab/Drive session.
Uploads are transfer states, not legal verification. No SQLite or public source mutation.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests>=2.32,<3", "beautifulsoup4", "google-api-python-client", "google-auth-httplib2"])


In [ ]:
"""Shared archive engine. Colab notebooks embed this file so they are self-contained.

Boundaries: raw evidence backup to private Drive, never policy-DB writes or public republication.
"""
from __future__ import annotations
import csv
import hashlib
import io
import json
import os
import re
import sqlite3
import tempfile
import time
import zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse, urlsplit, urlunsplit, unquote

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

GCCSI_CATALOG_ID = '1Vh_cLBzI1skRBXqbqe4fv-oiycuOWDxPzVdORs9RfZQ'
GCCSI_FOLDER_ID = '14oNNEBFEbx9piny8XdhUjpe3REcC8sgl'
POLICY_FOLDER_ID = '1S07jkytGKENRvA7zwXjnGARiKVZr5Vnx'
METADATA_FOLDER_ID = '12kOSesAeX367tV4gsx7GfH6JY_Q3X3Lo'
GITHUB_DB_URL = 'https://raw.githubusercontent.com/liuh886/ccus-policy-hub/main/agent/ccus-ai-agent/db/ccus_master.sqlite'

ROOT = Path('/content/ccus_archive_runtime') if Path('/content').exists() else Path(tempfile.gettempdir()) / 'ccus_archive_runtime'
MAX_BYTES = 300 * 1024 * 1024
RECHECK_DAYS = 45  # An existing file older than this can be checked again for updated bytes.
REQUEST_TIMEOUT = (10, 35)  # connection and socket read: retries happen on later incremental runs
PUBLISHER_DELAY_SECONDS = 0.6
USER_AGENT = 'CCUSPolicyHubSourceArchiver/3.0 (research source preservation; low request rate)'


def utcnow():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')


def sha_text(value):
    return hashlib.sha256(value.encode('utf-8')).hexdigest()


def normalize_url(url):
    if not isinstance(url, str):
        return None
    url = url.strip().rstrip(').,;]')
    if not url or len(url) > 2048:
        return None
    p = urlsplit(url)
    if p.scheme.lower() != 'https' or not p.hostname or p.username or p.password:
        return None
    return urlunsplit(('https', p.netloc.lower(), p.path or '/', p.query, ''))


def safe_filename(name):
    name = unquote(name or 'document').replace('\\', '/')
    name = name.split('/')[-1]
    name = re.sub(r'[^\w.()\- ]+', '_', name, flags=re.UNICODE).strip(' ._')[:145]
    return name or 'document'


def source_name(url, kind, policy_id=''):
    stem = safe_filename(urlsplit(url).path.rsplit('/', 1)[-1])
    if not re.search(r'\.(pdf|docx?|html?|txt)$', stem, re.I):
        stem += '.html'
    if kind == 'primary_policy':
        return safe_filename(policy_id)[:65] + '__' + stem
    return stem


def session_factory():
    s = requests.Session()
    s.headers.update({'User-Agent': USER_AGENT, 'Accept': 'application/pdf,text/html,application/vnd.openxmlformats-officedocument.wordprocessingml.document,*/*;q=0.7'})
    retry = Retry(total=1, connect=1, read=0, status=1, backoff_factor=1.5, status_forcelist=[429,500,502,503,504], allowed_methods=['GET'], respect_retry_after_header=True)
    s.mount('https://', HTTPAdapter(max_retries=retry, pool_connections=3, pool_maxsize=3))
    return s


def colab_services():
    from google.colab import auth
    import google.auth
    from googleapiclient.discovery import build
    from google_auth_httplib2 import AuthorizedHttp
    import httplib2
    auth.authenticate_user()
    creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive', 'https://www.googleapis.com/auth/documents.readonly'])
    drive_http = AuthorizedHttp(creds, http=httplib2.Http(timeout=90))
    docs_http = AuthorizedHttp(creds, http=httplib2.Http(timeout=90))
    return build('drive', 'v3', http=drive_http, cache_discovery=False), build('docs','v1',http=docs_http,cache_discovery=False)


def list_folder(drive, folder_id):
    result=[]; token=None
    while True:
        page = drive.files().list(q=f"'{folder_id}' in parents and trashed = false", pageSize=1000,
            fields='nextPageToken,files(id,name,mimeType,size,createdTime,modifiedTime,appProperties,description)',pageToken=token).execute()
        result.extend(page.get('files', []))
        token=page.get('nextPageToken')
        if not token: break
    return result


def existing_indices(files):
    by_url={}; by_sha={}; filenames=set()
    for obj in sorted(files, key=lambda v: v.get('createdTime','')):
        props=obj.get('appProperties') or {}
        h=props.get('source_url_sha256')
        sha=props.get('sha256')
        if h: by_url[h]=obj
        for k,v in props.items():
            if k.startswith('source_url_alias_') and len(v)==64:by_url[v]=obj
        if sha: by_sha[sha]=obj
        filenames.add(obj.get('name',''))
    return by_url,by_sha,filenames


def due_for_recheck(obj, days):
    if days is None: return False
    if days <= 0: return True
    props=obj.get('appProperties') or {}
    stamp=props.get('checked_at') or obj.get('createdTime') or obj.get('modifiedTime')
    if not stamp: return False  # unknown legacy timestamps: skip instead of redownloading all
    try:
        stamp=datetime.fromisoformat(stamp.replace('Z','+00:00'))
        return (datetime.now(timezone.utc)-stamp).total_seconds() >= days*86400
    except ValueError:
        return False


def blob_type(filepath, first, content_type, allow_html):
    if first.startswith(b'%PDF-'): return ('pdf','application/pdf')
    if first.startswith(b'PK\x03\x04') and zipfile.is_zipfile(filepath):
        with zipfile.ZipFile(filepath) as z:
            if 'word/document.xml' in z.namelist(): return ('docx','application/vnd.openxmlformats-officedocument.wordprocessingml.document')
    if allow_html and (b'<!doctype html' in first.lower() or b'<html' in first.lower() or 'text/html' in content_type):
        # Do not preserve access-control, CAPTCHA, generic error or bot-check pages as authoritative text.
        data=filepath.read_bytes()[:120000].lower()
        blocked=(b'captcha', b'cf-chl-', b'just a moment', b'access denied', b'please enable javascript', b'cloudflare ray id', b'type="password"', b'log in to view', b'sign in to continue')
        if any(x in data for x in blocked): return ('blocked_html',None)
        if len(data) < 700: return ('html_too_short',None)
        return ('html','text/html')
    return ('not_document',None)


def retrieve(session,url,workdir,allow_html=False,host_validator=None,headers=None):
    """Return (local path, info dict). Any status other than downloaded leaves no local file."""
    tmp=workdir/(sha_text(url)[:20]+'.part'); tmp.parent.mkdir(parents=True,exist_ok=True)
    info={'http_status':'','status':'','bytes':0,'sha256':'','format':'','etag':'','last_modified':'','resolved_url':'','error':''}
    try:
        with session.get(url, stream=True, timeout=REQUEST_TIMEOUT, allow_redirects=True, headers=headers or {}) as res:
            info['http_status']=res.status_code; info['resolved_url']=res.url
            info['etag']=(res.headers.get('ETag') or '')[:115]
            info['last_modified']=(res.headers.get('Last-Modified') or '')[:115]
            if res.status_code==304:
                info['status']='not_modified';return None,info
            if res.status_code in (401,403,407,412,419,429):
                info['status']='access_blocked';return None,info
            if res.status_code in (404,410):
                info['status']='not_found';return None,info
            if res.status_code==202:
                info['status']='http_202_not_document';return None,info
            if res.status_code!=200:
                info['status']='http_error';return None,info
            if host_validator and not host_validator(res.url):
                info['status']='redirect_outside_official';return None,info
            try: length=int(res.headers.get('Content-Length') or '0')
            except ValueError:length=0
            if length>MAX_BYTES:
                info['status']='too_large';return None,info
            digest=hashlib.sha256();total=0;first=b''
            with tmp.open('wb') as f:
                for block in res.iter_content(chunk_size=1024*1024):
                    if not block: continue
                    total+=len(block)
                    if total>MAX_BYTES:
                        info['status']='too_large';return None,info
                    if len(first)<1024:first=(first+block)[:1024]
                    digest.update(block);f.write(block)
            fmt,mime=blob_type(tmp,first,(res.headers.get('Content-Type') or '').lower(),allow_html)
            if not mime:
                info['status']=fmt;return None,info
            if total<500:
                info['status']='too_short';return None,info
            info.update(status='downloaded',bytes=total,sha256=digest.hexdigest(),format=fmt,mime_type=mime)
            return tmp,info
    except requests.RequestException as exc:
        info['status']='network_error';info['error']=f'{type(exc).__name__}: {str(exc)[:200]}'
        return None,info
    except Exception as exc:
        info['status']='validation_error';info['error']=f'{type(exc).__name__}: {str(exc)[:200]}'
        return None,info
    finally:
        if info['status']!='downloaded' and tmp.exists():tmp.unlink()


def create_upload(drive, local_file, folder_id, name, url, sha256, policy_id, kind, mime_type, headers, provenance=None):
    from googleapiclient.http import MediaFileUpload
    meta={'sha256':sha256,'source_url_sha256':sha_text(url),'checked_at':utcnow()}
    if policy_id: meta['policy_id']=str(policy_id)[:90]
    provenance=provenance or {}
    for prop in ('source_edition','target_origin'):
        if provenance.get(prop):meta[prop]=str(provenance[prop])[:120]
    if headers.get('etag'): meta['etag']=headers['etag']
    if headers.get('last_modified'): meta['last_modified']=headers['last_modified']
    media=MediaFileUpload(str(local_file),mimetype=mime_type,resumable=True,chunksize=4*1024*1024)
    return drive.files().create(body={'name':name,'parents':[folder_id],
             'description':f'RAW {kind} source. URL: {url}; SHA256: {sha256}. Not approved for public redistribution.',
             'appProperties':meta},media_body=media,fields='id,name,mimeType,size,createdTime,appProperties').execute()


def mark_checked(drive,obj,headers):
    props=dict(obj.get('appProperties') or {})
    props['checked_at']=utcnow()
    if headers.get('etag'):props['etag']=headers['etag']
    if headers.get('last_modified'):props['last_modified']=headers['last_modified']
    drive.files().update(fileId=obj['id'],body={'appProperties':props},fields='id,appProperties').execute()
    obj['appProperties']=props


def url_scope(url):
    """URL-level specificity only; never claim the file is the named instrument."""
    if not url or not normalize_url(url): return 'invalid_or_non_https'
    p=urlsplit(url)
    path=p.path.strip('/')
    if not path or path.lower() in ('en','en/','english','home','index.html','index.htm','web','about'):
        return 'site_homepage'
    if re.search(r'\.(pdf|docx|doc|html?|shtml|aspx)$', path, re.I):
        return 'specific_document_or_page_candidate'
    if len(path.split('/'))<=1:
        return 'section_or_landing_page'
    return 'specific_page_candidate'


def policy_source_relevance_status(url, kind):
    if kind!='primary_policy': return 'not_applicable'
    scope=url_scope(url)
    if scope=='site_homepage': return 'not_an_instrument'
    if scope=='invalid_or_non_https': return 'url_needs_repair'
    return 'needs_instrument_content_review'


def capture_targets(drive,session,folder_id,targets,kind,recheck_days=RECHECK_DAYS,max_new=None):
    """Idempotent: incremental discovery, stale recheck, content dedup, immutable versions."""
    by_url,by_sha,names=existing_indices(list_folder(drive,folder_id))
    result=[];seen=set();new_count=0
    workdir=ROOT/kind;workdir.mkdir(parents=True,exist_ok=True)
    for position, item in enumerate(targets,1):
        url=normalize_url(item.get('url'))
        row={'collected_at':utcnow(),'policy_id':item.get('policy_id',''),
             'country':item.get('country',''),'title':item.get('title',''),
             'publisher':item.get('publisher',''),'source_type':kind,'inventory_sha256':item.get('inventory_sha256',''), 'url':url or item.get('url',''),
             'status':'','http_status':'','format':'','bytes':'','sha256':'',
             'drive_file_id':'','resolved_url':'','error':'',
             'url_scope':url_scope(url or item.get('url','')),
             'target_origin':item.get('target_origin','canonical_db'),
             'source_edition':item.get('source_edition',''),
             'instrument_verification':policy_source_relevance_status(url or item.get('url',''),kind)}
        if not url:
            original=(item.get('url') or '').strip()
            status=('non_https_source' if original.lower().startswith('http://') else
                    'missing_source_url' if kind=='primary_policy' and not original else 'invalid_url')
            row.update(status=status,error='Preserved original URL; manual HTTPS verification required' if status=='non_https_source' else '')
            result.append(row);continue
        if (item.get('policy_id',''),url) in seen:continue
        seen.add((item.get('policy_id',''),url))
        # Broad ministry homepages are useful pointers, but not a policy's original text.
        # Never let HTTP 200 on a generic homepage inflate primary-evidence coverage.
        if kind=='primary_policy' and row['url_scope']=='site_homepage':
            row.update(status='source_too_broad',error='Institution homepage, not an identifiable policy instrument')
            result.append(row);continue
        if kind=='primary_policy' and not is_official_url(url):
            row.update(status='needs_official_source_review',error='Host not in conservative government/institutional allowlist');result.append(row);continue
        old=by_url.get(sha_text(url))
        if old and not due_for_recheck(old,recheck_days):
            props=old.get('appProperties') or {}
            row.update(status='already_archived',drive_file_id=old['id'],sha256=props.get('sha256',''),bytes=old.get('size',''))
            result.append(row);continue
        if max_new is not None and new_count>=max_new and old is None:
            row.update(status='new_item_limit');result.append(row);continue
        head={}
        if old:
            props=old.get('appProperties') or {}
            if props.get('etag'):head['If-None-Match']=props['etag']
            if props.get('last_modified'):head['If-Modified-Since']=props['last_modified']
        local,info=retrieve(session,url,workdir,allow_html=(kind=='primary_policy'),
                            host_validator=is_official_url if kind=='primary_policy' else is_gccsi_url, headers=head)
        row.update({k:info.get(k,'') for k in ('http_status','format','bytes','sha256','resolved_url','error')})
        row['status']=info['status']
        try:
            if info['status']=='not_modified' and old:
                try:mark_checked(drive,old,info)
                except Exception as e:row['error']='mark_checked: '+str(e)[:180]
                row.update(status='not_modified',drive_file_id=old['id'],sha256=(old.get('appProperties') or {}).get('sha256',''))
            elif local:
                if old and (old.get('appProperties') or {}).get('sha256')==info['sha256']:
                    try:mark_checked(drive,old,info)
                    except Exception as e:row['error']='mark_checked: '+str(e)[:180]
                    row.update(status='unchanged_content',drive_file_id=old['id'])
                elif info['sha256'] in by_sha:
                    existing=by_sha[info['sha256']]
                    row.update(status='duplicate_content',drive_file_id=existing['id'])
                    # Persist alias for future runs, otherwise a deduplicated URL would be re-downloaded forever.
                    alias='source_url_alias_'+sha_text(url)[:14]
                    try:
                        props=dict(existing.get('appProperties') or {})
                        props[alias]=sha_text(url)
                        drive.files().update(fileId=existing['id'],body={'appProperties':props},fields='id,appProperties').execute()
                        existing['appProperties']=props
                        by_url[sha_text(url)]=existing
                    except Exception as exc:
                        row['error']='alias_persistence_failed: '+str(exc)[:160]
                else:
                    basename=source_name(url,kind,row['policy_id'])
                    # Actual source format is authoritative even if the landing URL has a misleading extension.
                    stem,ext=os.path.splitext(basename)
                    expected_ext='.'+info['format']
                    if ext.lower()!=expected_ext:
                        basename=stem+expected_ext
                    if basename in names:
                        stem,ext=os.path.splitext(basename)
                        basename=f'{stem}__{info["sha256"][:12]}{ext}'
                    try:
                        created=create_upload(drive,local,folder_id,basename,url,info['sha256'],row['policy_id'],kind,info['mime_type'],info,row)
                        row.update(status='uploaded_original',drive_file_id=created['id'])
                        by_url[sha_text(url)]=created;by_sha[info['sha256']]=created;names.add(basename);new_count+=1
                    except Exception as exc:
                        row.update(status='drive_upload_failed',error=f'{type(exc).__name__}: {str(exc)[:220]}')
        finally:
            if local and local.exists():local.unlink()
        result.append(row)
        if position%10==0:print(f'{kind} {position}: {Counter(x["status"] for x in result)}')
        time.sleep(PUBLISHER_DELAY_SECONDS)
    return result


def export_audit(drive,rows,prefix):
    if not rows:raise ValueError('Empty audit')
    from googleapiclient.http import MediaFileUpload
    ROOT.mkdir(parents=True,exist_ok=True)
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    path=ROOT/f'{prefix}_Audit_{stamp}.csv'
    keys=['collected_at','policy_id','country','title','publisher','source_type','inventory_sha256','url','status','http_status',
          'format','bytes','sha256','drive_file_id','resolved_url','error','url_scope','instrument_verification','target_origin','source_edition']
    with path.open('w',encoding='utf-8-sig',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=keys,extrasaction='ignore');writer.writeheader();writer.writerows(rows)
    media=MediaFileUpload(str(path),mimetype='text/csv',resumable=True)
    created=drive.files().create(body={'name':path.name,'parents':[METADATA_FOLDER_ID],
         'description':f'Immutable per-run {prefix} source archive audit with successful, skipped and failed sources'},
         media_body=media,fields='id,name,webViewLink').execute()
    print(f'COMPLETE: {len(rows)} source attempts; statuses: {dict(Counter(r["status"] for r in rows))}')
    print(f'AUDIT: https://drive.google.com/file/d/{created["id"]}/view')
    return created


def is_gccsi_url(url):
    p=urlsplit(url)
    return p.scheme=='https' and (p.hostname or '').lower() in ('globalccsinstitute.com','www.globalccsinstitute.com')


# Conservative official-host policy. This is a starting review filter, not evidence of legal authority.
OFFICIAL_HOSTS={
 'eur-lex.europa.eu','ec.europa.eu','commission.europa.eu','op.europa.eu',
 'legislation.gov.uk','www.legislation.gov.uk','gov.uk','www.gov.uk',
 'regjeringen.no','www.regjeringen.no','lovdata.no','www.lovdata.no',
 'canada.ca','www.canada.ca','laws-lois.justice.gc.ca','gazette.gc.ca',
 'ato.gov.au','legislation.gov.au','www.legislation.gov.au',
 'austlii.edu.au','www.austlii.edu.au','govinfo.gov','www.govinfo.gov',
 'gov.cn','www.gov.cn','mee.gov.cn','www.mee.gov.cn',
 'english.www.gov.cn','mof.gov.cn','ndrc.gov.cn',
 'meti.go.jp','www.meti.go.jp','e-gov.go.jp','elaws.e-gov.go.jp',
 'law.go.kr','www.law.go.kr','moleg.go.kr','www.moleg.go.kr',
 'gov.sg','www.gov.sg','sso.agc.gov.sg',
 'wipo.int','www.wipo.int','unfccc.int','www.unfccc.int',
 'alberta.ca','www.alberta.ca','kings-printer.alberta.ca',
 'saskatchewan.ca','www.saskatchewan.ca',
 'ens.dk','www.ens.dk','wetten.overheid.nl',
 'u.ae','www.u.ae','www.bmwk.de','www.government.nl',
 'www.rvo.nl','english.rvo.nl','www.sodir.no',
 'www25.senado.leg.br','www.boletinoficial.gob.ar','www.althingi.is',
 'imo.org','www.imo.org','un.org','www.un.org',
}
OFFICIAL_SUFFIXES=('.gov','.gov.uk','.gov.cn','.gov.au','.gov.sg','.gov.ca','.gouv.fr','.gob.mx','.go.jp','.go.kr','.europa.eu','.gc.ca','.admin.ch','.gov.br','.gov.in','.gov.za','.gov.ie','.gov.no',
 '.gov.ae','.gov.qa','.gov.kz','.gov.pl','.gov.ph','.gov.tr','.gov.ro','.gov.sa','.gov.ng','.gov.eg','.gov.my','.gov.om','.gov.co',
 '.gob.ar','.gob.cl','.gob.es','.gob.pe','.go.th','.go.id','.gov.vn','.go.ug','.gov.gr')
EXTRA_OFFICIAL_HOSTS=set()  # Add hostnames only after manually verifying their source authority.

def is_official_url(url):
    p=urlsplit(url)
    if p.scheme!='https' or p.username or p.password:return False
    host=(p.hostname or '').lower()
    return (host in OFFICIAL_HOSTS or host in EXTRA_OFFICIAL_HOSTS
            or any(host.endswith(suffix) or host==suffix.lstrip('.') for suffix in OFFICIAL_SUFFIXES))


In [ ]:
"""Policy source acquisition and review queue (v4.1).

Run next to archive_core.py (GitHub) or ccus_archive_core.py (Google Drive).
The Colab notebook embeds both modules and needs neither import.
No changes to the canonical policy SQLite.
"""


# ==== COLLECTION-SPECIFIC MODE ====
"""CCUS Policy Hub primary policy evidence acquisition, read-only vs canonical SQLite.

Use the repository's policy IDs and stored source URLs to preserve real originals.
This does NOT search policy names for plausible-looking documents or mutate SQLite.
"""

POLICY_RECHECK_DAYS = 30
POLICY_MAX_NEW_PER_RUN = None
# Optional Drive CSV with headers: policy_id,country,title,url,publisher.
# Put verified regulator/legislature primary-source URLs in this manual source queue.
POLICY_ADDITIONAL_TARGETS_CSV_ID = ''
# An approved queue requires policy_id,url,review_status=approved,reviewer,source_edition.
# The discovery CSV is NOT an approved-targets file; never pass it here without review.
POLICY_REVIEW_CANDIDATES = True
POLICY_RESEARCH_BACKLOG = True
MAX_HTML_SNAPSHOTS_TO_SCAN = 55
MAX_HTML_SOURCE_BYTES = 3500000
MAX_CANDIDATES_PER_POLICY = 6


def fetch_canonical_policies(session):
    ROOT.mkdir(parents=True,exist_ok=True)
    path=ROOT/'canonical_policy_snapshot.sqlite'
    r=session.get(GITHUB_DB_URL,stream=True,timeout=(15,120))
    r.raise_for_status()
    size=0
    with path.open('wb') as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            if not chunk:continue
            size+=len(chunk)
            if size>80*1024*1024:raise RuntimeError('Unexpectedly oversized SQLite source')
            f.write(chunk)
    with path.open('rb') as f:
        if f.read(16)!=b'SQLite format 3\x00':raise RuntimeError('GitHub response is not a SQLite database; stop')
    inventory_sha=hashlib.sha256(path.read_bytes()).hexdigest()
    print('Canonical policy DB SHA-256:',inventory_sha)
    db=sqlite3.connect('file:'+str(path)+'?mode=ro',uri=True)
    try:
        chk=db.execute('PRAGMA integrity_check').fetchone()
        if not chk or chk[0]!='ok':raise RuntimeError('Canonical SQLite integrity check failed')
        records=db.execute('''
          SELECT p.id,p.country,COALESCE(p.url,''),COALESCE(p.source,''),
                 COALESCE((SELECT i.title FROM policy_i18n i WHERE i.policy_id=p.id AND i.lang='en' LIMIT 1),p.id)
          FROM policies p ORDER BY p.id
        ''').fetchall()
    finally:
        db.close()
    if not records:raise RuntimeError('Canonical database returned zero policy records; stop')
    out=[]
    for policy_id,country,url,issuer,title in records:
        # 'source' is normally a citation/publisher, not a URL; never fabricate one.
        primary=url.strip() if url else ''
        if not primary and issuer.lower().startswith('https://'):primary=issuer.strip()
        out.append({'policy_id':policy_id,'country':country,'url':primary,'title':title,'publisher':issuer,'inventory_sha256':inventory_sha})
    print('Canonical policy records:',len(out), 'with nonempty URL:',sum(bool(x['url']) for x in out),
          'with valid HTTPS URL:',sum(bool(normalize_url(x['url'])) for x in out))
    return out


def parse_approved_targets(reader, canonical_targets):
    required={'policy_id','url','review_status','reviewer','source_edition'}
    if not required.issubset(set(reader.fieldnames or [])):
        raise ValueError('Approved source CSV must have: '+', '.join(sorted(required)))
    ids={x['policy_id']:x for x in canonical_targets}
    data=[]
    for row in reader:
        policy_id=row.get('policy_id','').strip()
        if not policy_id:continue
        if policy_id not in ids:raise ValueError('Unknown policy ID in approval CSV: '+policy_id)
        if row.get('review_status','').strip().lower()!='approved':continue
        if not row.get('reviewer','').strip() or not row.get('source_edition','').strip():
            raise ValueError('Approved target needs reviewer + source_edition: '+policy_id)
        url=normalize_url(row.get('url',''))
        if not url or not is_official_url(url) or url_scope(url)=='site_homepage':
            raise ValueError('Approved target URL must be HTTPS, official and document-specific: '+policy_id)
        source=ids[policy_id]
        data.append({**source,'url':url,'target_origin':'manual_approved',
                     'source_edition':row['source_edition'].strip(),
                     'reviewer':row['reviewer'].strip()})
    return data


def manual_targets(drive, canonical_targets):
    """Only reviewed exact URLs, with policy IDs bound to canonical SQLite."""
    if not POLICY_ADDITIONAL_TARGETS_CSV_ID:return []
    from googleapiclient.http import MediaIoBaseDownload
    request=drive.files().get_media(fileId=POLICY_ADDITIONAL_TARGETS_CSV_ID)
    buf=io.BytesIO();downloader=MediaIoBaseDownload(buf,request)
    done=False
    while not done:_,done=downloader.next_chunk()
    reader=csv.DictReader(io.StringIO(buf.getvalue().decode('utf-8-sig')))
    return parse_approved_targets(reader,canonical_targets)


# Official verified reference candidates, kept in **review queue only**.
# They are NOT downloaded or inserted into SQLite automatically.
OFFICIAL_RESEARCH_SEEDS = [
    {'policy_id':'uk-energy-act-2023',
     'candidate_url':'https://www.legislation.gov.uk/ukpga/2023/52/pdfs/ukpga_20230052_en.pdf',
     'candidate_title':'Energy Act 2023 — chapter 52 (original enacted PDF)',
     'candidate_type':'original_enacted_legislation','source_edition':'original_enacted',
     'research_note':'Corrects invalid canonical 2023/67; confirm the required original vs current consolidated edition.'},
    {'policy_id':'us-epa-class-vi-primacy',
     'candidate_url':'https://www.epa.gov/uic/primary-enforcement-authority-underground-injection-control-program',
     'candidate_title':'US EPA UIC primacy programme explanation',
     'candidate_type':'regulator_guidance','source_edition':'current_webpage',
     'research_note':'Supporting context only. Not the statute, 40 CFR regulation or individual primacy approval.'},
    {'policy_id':'us-cifia-lpo',
     'candidate_url':'https://www.energy.gov/edf/carbon-dioxide-transportation-infrastructure-finance-and-innovation-program',
     'candidate_title':'US DOE CIFIA programme overview',
     'candidate_type':'programme_overview','source_edition':'current_webpage',
     'research_note':'Programme explanation, not legislative original. Do not reuse for us-iija-hubs without identity review.'},
]

from html.parser import HTMLParser
from urllib.parse import urljoin

class _SafeAnchorParser(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.links=[];self.href=None;self.caption=[]
    def handle_starttag(self, tag, attrs):
        if tag.lower()=='a':
            attrs=dict(attrs);self.href=attrs.get('href');self.caption=[]
    def handle_data(self, data):
        if self.href is not None and len(' '.join(self.caption))<280:self.caption.append(data)
    def handle_endtag(self, tag):
        if tag.lower()=='a' and self.href is not None:
            self.links.append((self.href,' '.join(' '.join(self.caption).split())[:240]))
            self.href=None;self.caption=[]


# Keep generic organization/financial words out of the title-identity evidence.
STOP_TOKENS={'carbon','capture','storage','ccs','ccus','climate','change','energy','the','and',
    'policy','scheme','national','government','for','with','framework','programme','program',
    'report','of','act','law','regulation','directive','decree','legislation','bill','rules',
    'statute','brazil','brazilian','india','indian','united','states','british','united',
    'federal','agency','authority','ministry','resolution','tax','credit','incentive',
    'anp','irs','epa','doe','government','net','zero','2024','2025','2026','2023',
    '2022','2021','2020','2019','2027','2028','2029','2030','original','page'}
IRRELEVANT_LINK_RE = re.compile(
    r'(?i)(?:privacy|cookie|lgpd|governan[cç]a.{0,12}privacidade|termo[-_ ]?uso|'
    r'tax[-_ ]fraud|tax fraud|scam|child[-_ ]tax|child tax credit|earned[-_ ]income|'
    r'budget_at_glance|transfer.of.resources|contingency.fund|deficit.statistics|'
    r'/report-fraud/|governanca-privacidade)')
NAVIGATION_LINK_RE=re.compile(r'(?i)(?:expert.group|innovation.fund.projects|key.insights|calls.for.proposals|competitive.bidding|battery.booster)')
SUBJECT_GUARDS = {
    'br-anp-ccs-resolution': re.compile(r'(?i)(?:ccs|ccus|co2|carbon[o]?|captur[ae]|armazenamento|sequestro|14[._ -]?993)'),
    'us-45q-ira': re.compile(r'(?i)(?:45[ _-]?q|carbon.{0,24}(?:sequest|capture)|(?:sequest|capture).{0,24}carbon)'),
    'in-union-budget-2026-ccus': re.compile(r'(?i)(?:ccus|ccs|co2|carbon.{0,24}captur|captur.{0,24}carbon)'),
}


def _title_terms(value):
    return {v for v in re.findall(r'[a-z0-9]{3,}', (value or '').lower())
            if v not in STOP_TOKENS and not v.isdigit()}


def candidate_kind(url, anchor):
    content=(urlsplit(url).path+' '+anchor).lower()
    if re.search(r'\.(pdf|docx?)$',urlsplit(url).path,re.I):return 'document_file_candidate'
    if re.search(r'\b(act|statute|regulation|directive|decree|ordinance|legislation|law|code)\b', content):
        return 'legal_text_candidate'
    if re.search(r'\b(grant|subsidy|fund|programme|program|incentive|guidance)\b',content):
        return 'programme_or_guidance_candidate'
    return 'unspecified_webpage_candidate'


def candidate_version_alignment(policy, candidate_url, anchor):
    """Heuristic labels only; cannot establish enactment/consolidation/legal effect."""
    name=policy.get('title','')
    raw=' '.join([anchor or '',urlsplit(candidate_url).path])
    years=re.findall(r'(?<!\d)(?:19|20)\d{2}(?!\d)', name)
    years_seen=set(re.findall(r'(?<!\d)(?:19|20)\d{2}(?!\d)', raw))
    if not years:return 'no_policy_year'
    target=years[-1]
    if target in years_seen:return 'year_token_match_unverified'
    if years_seen and max(years_seen)<target:return 'earlier_document_year_review'
    if years_seen:return 'other_document_year_review'
    return 'publication_year_not_visible'


# These are candidate-stage descriptions, never legal identity/version approval.
LANGUAGE_SELECTOR_RE = re.compile(r'(?i)^\s*(?:bg|cs|da|de|el|en|es|et|fi|fr|ga|hr|hu|it|lt|lv|mt|nl|pl|pt|ro|sk|sl|sv)\s+[^/]{2,45}$')

def candidate_relation(policy, candidate_url, anchor, base_url=None):
    caption=' '.join((anchor or '').split())
    parsed=urlsplit(candidate_url)
    if LANGUAGE_SELECTOR_RE.fullmatch(caption):
        return 'navigation'
    if re.search(r'(?i)(?:select[-_ /]?language|language[-_ /]?selector)', parsed.path+' '+caption):
        return 'navigation'
    # Same registry identity with another internal HTML route is not a new Act.
    original=urlsplit(base_url or policy.get('url',''))
    if parsed.hostname==original.hostname and parsed.hostname in ('www.legislation.gov.au','legislation.gov.au'):
        identity=lambda path: re.search(r'(?i)/(C\d{4}[AC]\d+|F\d{4}[LC]\d+)(?:/|$)',path)
        left,right=identity(original.path),identity(parsed.path)
        if left and right and left[1].casefold()==right[1].casefold():
            return 'same_instrument_internal_section'
    if policy.get('policy_id')=='uk-industrial-carbon-capture-business-model-2025':
        text=caption+' '+parsed.path.rsplit('/',1)[-1]
        if not re.search(r'(?i)(?:industrial[ _-]+carbon[ _-]+capture|\bICC\b)',text):
            return 'related_secondary' if re.search(r'(?i)(?:dispatchable|transport|storage|business.model)',text) else 'unknown'
    if candidate_version_alignment(policy,candidate_url,anchor)=='earlier_document_year_review':
        return 'historical_version'
    title=' '.join(policy.get('title','').casefold().split())
    if len(title)>=12 and title in caption.casefold():
        return 'exact_instrument'
    return 'unknown'


def score_candidate(policy, candidate_url, anchor):
    """Candidate ranking only. Require identity clues; never use as legal authority score."""
    if not is_official_url(candidate_url) or url_scope(candidate_url)=='site_homepage':return -1
    if candidate_relation(policy,candidate_url,anchor)=='navigation':return -1
    path=urlsplit(candidate_url).path
    # Do not award title matches from a long shared portal prefix (EU Innovation Fund, etc.).
    # Only the anchor and the last URL path component are candidate-specific.
    context=' '.join((anchor or '',path.rstrip('/').rsplit('/',1)[-1]))
    if NAVIGATION_LINK_RE.search(context) and not NAVIGATION_LINK_RE.search(policy.get('title','')):return -1
    if IRRELEVANT_LINK_RE.search(context) and not IRRELEVANT_LINK_RE.search(policy.get('title','')):
        return -1
    # An exact named instrument in the anchor is stronger than generic token overlap.
    # Important for concise titles such as Energy Act 2023 (whose words are otherwise common).
    exact_title=' '.join(policy.get('title','').casefold().split())
    anchor_title=' '.join((anchor or '').casefold().split())
    if len(exact_title)>=12 and exact_title in anchor_title:
        edition=candidate_version_alignment(policy,candidate_url,anchor)
        return 83 if edition!='earlier_document_year_review' else 45
    guard=SUBJECT_GUARDS.get(policy.get('policy_id'))
    if guard and not guard.search(context):return -1
    matched=_title_terms(policy.get('title','')) & _title_terms(context)
    kind=candidate_kind(candidate_url,anchor)
    # A generic document at the correct publisher is not evidence of a specific policy.
    if len(matched)<2:
        # Specific instrument identifiers, where supplied in the canonical title, may substitute for words.
        identifiers=re.findall(r'(?<!\d)\d{2,5}(?:[.-]\d{1,5})?(?!\d)',policy.get('title',''))
        identifiers=[x for x in identifiers if x not in {'2024','2025','2026','2023','2022','2021','2020'}]
        exact=bool(identifiers and any(i in context for i in identifiers))
        if not exact:return -1
    score=min(65,len(matched)*18)
    if kind=='document_file_candidate':score+=12
    if kind=='legal_text_candidate':score+=9
    if kind=='programme_or_guidance_candidate':score+=3
    if urlsplit(candidate_url).hostname==urlsplit(policy.get('url','')).hostname:score+=3
    edition=candidate_version_alignment(policy,candidate_url,anchor)
    if edition=='earlier_document_year_review':score-=25
    elif edition=='other_document_year_review':score-=15
    elif edition=='year_token_match_unverified':score+=6
    return min(95,score) if score>=20 else -1


def extract_official_link_candidates(html_bytes,base_url,policy,source_file_id,max_candidates=6):
    """Read already-archived HTML once, do not crawl or fetch linked sites."""
    if not base_url or not is_official_url(base_url):return []
    parser=_SafeAnchorParser()
    try:parser.feed(html_bytes.decode('utf-8',errors='replace'))
    except Exception:return []
    results={}
    for href,anchor in parser.links[:2500]:
        if not href or len(href)>1600:continue
        url=normalize_url(urljoin(base_url,href))
        if not url or url==normalize_url(base_url):continue
        # Anchor suggestions are for human review; neither cross-domain navigation nor SSL failure may trigger downloads.
        score=score_candidate(policy,url,anchor)
        if score<20:continue
        # A law may reside on a different official publisher host, but still requires manual verification.
        candidate={'policy_id':policy['policy_id'],'country':policy.get('country',''),
                   'policy_title':policy.get('title',''),'canonical_url':policy.get('url',''),
                   'candidate_url':url,'candidate_title':anchor,
                   'candidate_type':candidate_kind(url,anchor),
                   'source_edition':'unknown','source_origin':'archived_official_html_link',
                   'source_drive_file_id':source_file_id,'match_score':score,
                   'version_alignment':candidate_version_alignment(policy,url,anchor),
                   'candidate_relation':candidate_relation(policy,url,anchor,base_url),
                   'relation_review_status':'not_reviewed',
                   'review_status':'pending','evidence_match_status':'unverified',
                   'research_note':'Link extracted from archived official HTML; existence and identity not checked.',
                   'inventory_sha256':policy.get('inventory_sha256','')}
        if url not in results or results[url]['match_score']<score:results[url]=candidate
    return sorted(results.values(),key=lambda r:(-r['match_score'],r['candidate_url']))[:max_candidates]


def policy_archive_state(policy_id, rows, files):
    """Archive transfer and candidate discovery are separate dimensions."""
    audit=[r for r in rows if r.get('policy_id')==policy_id and
           r.get('target_origin','canonical_db')=='canonical_db']
    current=audit[-1] if audit else {}
    attached=[f for f in files if (f.get('appProperties') or {}).get('policy_id')==policy_id]
    # Reviewed manual batch imports intentionally have no Drive appProperties.
    # Their immutable SHA-prefixed names are a discovery hint, NOT legal verification.
    # The separate validated acquisition manifest binds file ID, URL and full SHA-256.
    manual=[f for f in files if re.fullmatch(re.escape(policy_id)+
             r'__[0-9a-f]{14}\.(?:pdf|html)', f.get('name',''))]
    status=current.get('status','unknown')
    if manual:
        return {'archive_state':'raw_candidate_archived',
                'archived_source_file_id':manual[0].get('id',''),
                'source_role':'manual_source_candidate_manifest_review_needed',
                'archive_audit_status':status}
    fid=current.get('drive_file_id') or (attached[0].get('id') if attached else '')
    if not fid:return {'archive_state':'no_raw_file','archived_source_file_id':'',
                       'source_role':'not_assessed','archive_audit_status':status}
    # A historical ministry-homepage snapshot must remain identifiable but not count as text.
    if status=='source_too_broad':
        return {'archive_state':'raw_homepage_only','archived_source_file_id':fid,
                'source_role':'institutional_context','archive_audit_status':status}
    return {'archive_state':'raw_candidate_archived','archived_source_file_id':fid,
            'source_role':'unverified_raw_source','archive_audit_status':status}


def publisher_lookup_route(policy):
    """An agent-ready research *hint*, never an invented search result or fetched URL."""
    from urllib.parse import urlsplit
    url=normalize_url(policy.get('url',''))
    host=urlsplit(url).hostname if url else ''
    country=(policy.get('country') or '').lower()
    if country=='european union' or host in ('eur-lex.europa.eu','op.europa.eu'):
        return 'eurlex_celex_or_eli_exact_identifier'
    if country=='united kingdom' or host.endswith('legislation.gov.uk'):
        return 'uk_legislation_exact_act_year_chapter'
    if country=='netherlands' or host.endswith('overheid.nl'):
        return 'nl_official_legislation_regulation_identifier'
    if country=='united states':
        return 'us_govinfo_ecfr_state_register_by_instrument'
    if country=='china':
        return 'cn_official_gazette_document_number_and_issuer'
    return 'issuer_official_registry_exact_title_number'


def make_research_backlog(canonical, rows, files, candidate_rows):
    by_id={r['policy_id']:r for r in canonical}
    number={}
    for r in candidate_rows:
        if r.get('candidate_url'):number[r['policy_id']]=number.get(r['policy_id'],0)+1
    result=[]
    for policy_id,policy in by_id.items():
        status=policy_archive_state(policy_id,rows,files)
        count=number.get(policy_id,0)
        instrument_gap=status['archive_state'] in ('no_raw_file','raw_homepage_only')
        priority=('P0' if instrument_gap and
                  status['archive_audit_status'] in ('source_too_broad','not_found','non_https_source')
                  else 'P1' if instrument_gap else 'P2')
        search_host=urlsplit(normalize_url(policy.get('url','')) or '').hostname or ''
        result.append({'policy_id':policy_id,'country':policy.get('country',''),
                       'policy_title':policy.get('title',''),'publisher':policy.get('publisher',''),
                       'canonical_url':policy.get('url',''),**status,
                       'source_candidate_count':count,
                       'discovery_state':'suggestions_pending_review' if count else
                          'homepage_only_exact_source_lookup_needed' if status['archive_state']=='raw_homepage_only' else
                          'no_new_link_but_archived_raw' if status['archive_state']!='no_raw_file'
                          else 'exact_source_lookup_needed',
                       'instrument_match_review_state':'not_reviewed',
                       'research_priority':priority,'publisher_lookup_route':publisher_lookup_route(policy),
                       'publisher_domain_hint':search_host,
                       'research_query':policy.get('title','')+' | '+policy.get('publisher',''),
                       'inventory_sha256':policy.get('inventory_sha256','')})
    return sorted(result,key=lambda r:(r['research_priority'],r['policy_id']))


def review_candidates_from_archives(drive,canonical,rows,files,read_limit=MAX_HTML_SNAPSHOTS_TO_SCAN):
    """Review-only: one row per policy minimum, with evidence state on every row."""
    by_id={p['policy_id']:p for p in canonical}
    output=[];covered=set();seen_file=set()
    for file in sorted(files,key=lambda f:f.get('name','')):
        props=file.get('appProperties') or {}
        policy_id=props.get('policy_id','')
        if policy_id not in by_id or file.get('mimeType')!='text/html':continue
        if len(seen_file)>=read_limit:break
        size=int(file.get('size') or 0)
        if size<500 or size>MAX_HTML_SOURCE_BYTES:continue
        fid=file.get('id')
        if fid in seen_file:continue
        seen_file.add(fid)
        description=file.get('description') or ''
        m=re.search(r'\bURL: (https://[^;\s]+); SHA256:', description)
        base_url=m.group(1) if m else by_id[policy_id]['url']
        if not is_official_url(base_url):continue
        try:
            page=drive.files().get_media(fileId=fid).execute()
            if not isinstance(page,bytes):continue
        except Exception:continue
        found=extract_official_link_candidates(page,base_url,by_id[policy_id],fid,MAX_CANDIDATES_PER_POLICY)
        output.extend(found);covered.update(r['policy_id'] for r in found)
    for seed in OFFICIAL_RESEARCH_SEEDS:
        policy=by_id.get(seed['policy_id'])
        if not policy:continue
        output.append({'policy_id':policy['policy_id'],'country':policy.get('country',''),
                       'policy_title':policy.get('title',''),'canonical_url':policy.get('url',''),
                       **seed,'source_origin':'official_research_seed',
                       'source_drive_file_id':'','match_score':'',
                       'version_alignment':candidate_version_alignment(policy,seed['candidate_url'],seed['candidate_title']),
                       'candidate_relation':candidate_relation(policy,seed['candidate_url'],seed['candidate_title']),
                       'relation_review_status':'not_reviewed',
                       'review_status':'pending','evidence_match_status':'unverified',
                       'inventory_sha256':policy.get('inventory_sha256','')})
        covered.add(policy['policy_id'])
    for policy in canonical:
        if policy['policy_id'] in covered:continue
        output.append({'policy_id':policy['policy_id'],'country':policy.get('country',''),
            'policy_title':policy.get('title',''),'canonical_url':policy.get('url',''),
            'candidate_url':'','candidate_title':'',
            'candidate_type':'no_new_link_candidate', 'source_edition':'',
            'source_origin':'research_backlog','source_drive_file_id':'',
            'match_score':'','review_status':'pending','evidence_match_status':'not_reviewed',
            'version_alignment':'not_applicable','candidate_relation':'unknown',
            'relation_review_status':'not_reviewed',
            'research_note':'No new outbound link discovered. Raw archive status is a separate field.',
            'inventory_sha256':policy.get('inventory_sha256','')})
    for row in output:
        state=policy_archive_state(row['policy_id'],rows,files)
        row.update(state)
        row['discovery_state']='suggestion_pending_review' if row.get('candidate_url') else (
            'no_new_link_but_archived_raw' if state['archive_state']!='no_raw_file' else 'exact_source_lookup_needed')
        row['instrument_match_review_state']='not_reviewed'
        row['publisher_lookup_route']=publisher_lookup_route(by_id[row['policy_id']])
    return sorted(output,key=lambda r:(r['policy_id'],not bool(r.get('candidate_url')),
                  -(int(r['match_score']) if str(r.get('match_score','')).isdigit() else 0),r.get('candidate_url','')))


def export_candidate_review_queue(drive,rows):
    from googleapiclient.http import MediaFileUpload
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    ROOT.mkdir(parents=True,exist_ok=True)
    file=ROOT/f'Policy_Source_Review_Candidates_{stamp}.csv'
    fields=['policy_id','country','policy_title','canonical_url','candidate_url','candidate_title',
        'candidate_type','source_edition','source_origin','source_drive_file_id','match_score',
        'review_status','evidence_match_status','research_note','inventory_sha256',
        'archive_state','archive_audit_status','archived_source_file_id','source_role',
        'discovery_state','instrument_match_review_state','version_alignment','candidate_relation','relation_review_status','publisher_lookup_route']
    with file.open('w',encoding='utf-8-sig',newline='') as out:
        writer=csv.DictWriter(out,fieldnames=fields,extrasaction='ignore')
        writer.writeheader();writer.writerows(rows)
    created=drive.files().create(body={'name':file.name,'parents':[METADATA_FOLDER_ID],
        'description':'Read-only evidence-aware source candidates. Always pending human review.'},
        media_body=MediaFileUpload(str(file),mimetype='text/csv',resumable=True),fields='id,name').execute()
    print(f'REVIEW QUEUE: {len(rows)} rows covering {len({x["policy_id"] for x in rows})} policy IDs; '
          f'https://drive.google.com/file/d/{created["id"]}/view')
    return created


def export_research_backlog(drive,records):
    from googleapiclient.http import MediaFileUpload
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    path=ROOT/f'Policy_Official_Registry_Research_Backlog_{stamp}.csv'
    fields=['policy_id','country','policy_title','publisher','canonical_url','archive_state',
        'archive_audit_status','archived_source_file_id','source_role','source_candidate_count',
        'discovery_state','instrument_match_review_state','research_priority',
        'publisher_lookup_route','publisher_domain_hint','research_query','inventory_sha256']
    with path.open('w',encoding='utf-8-sig',newline='') as out:
        w=csv.DictWriter(out,fieldnames=fields);w.writeheader();w.writerows(records)
    created=drive.files().create(body={'name':path.name,'parents':[METADATA_FOLDER_ID],
        'description':'Official registry research hints; no unverified URLs generated or fetched.'},
        media_body=MediaFileUpload(str(path),mimetype='text/csv',resumable=True),fields='id,name').execute()
    print(f'RESEARCH BACKLOG: {len(records)} policies; https://drive.google.com/file/d/{created["id"]}/view')
    return created


def main():
    drive,docs=colab_services();sess=session_factory()
    try:targets=fetch_canonical_policies(sess)
    except Exception as exc:
        raise RuntimeError('Canonical policy DB is unavailable: cannot infer missing sources; no archive attempt.') from exc
    approved=manual_targets(drive,targets)
    canonical=targets
    targets=canonical+approved
    rows=capture_targets(drive,sess,POLICY_FOLDER_ID,targets,'primary_policy',
                         recheck_days=POLICY_RECHECK_DAYS,max_new=POLICY_MAX_NEW_PER_RUN)
    export_audit(drive,rows,'Policy_Primary_Source')
    if POLICY_REVIEW_CANDIDATES:
        preserved=list_folder(drive,POLICY_FOLDER_ID)
        candidate_rows=review_candidates_from_archives(drive,canonical,rows,preserved)
        export_candidate_review_queue(drive,candidate_rows)
        if POLICY_RESEARCH_BACKLOG:
            export_research_backlog(drive,make_research_backlog(canonical,rows,preserved,candidate_rows))
    print('Primary evidence directory: https://drive.google.com/drive/folders/'+POLICY_FOLDER_ID)
    print('Important: source_too_broad / non_https_source / needs_official_source_review represent evidence gaps, not membership restrictions.')
    print('Any uploaded_original file is a raw archived source, not yet a legally verified policy instrument.')
    print('Review candidates are pending, never auto-downloaded or approved. No SQLite mutation or public republication.')



In [ ]:
main()
